In [1]:
%pip install -q "transformers[sentencepiece,torch]" evaluate sacremoses protobuf

# Import necessary packages
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
import evaluate

from transformers import (
    logging,
    pipeline,
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
)
logging.set_verbosity(logging.WARNING)


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\Victus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Removed forced kernel termination.
# Keep this cell as a no-op so the notebook runs end-to-end.

In [3]:
# Force a reload of the registry
from transformers.pipelines import SUPPORTED_TASKS

text = pd.read_csv("car_reviews.csv", sep=";")

classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

reviews = text["Review"].tolist()
predicted_labels = classifier(reviews, truncation=True)

predictions = [1 if x['label'] == 'POSITIVE' else 0 for x in predicted_labels]

y_true = text["Class"].map({"NEGATIVE": 0, "POSITIVE": 1})

accuracy_result = accuracy_score(y_true, predictions)
f1_result = f1_score(y_true, predictions)
print(accuracy_result, f1_result)

# Translating car reviews en-to-es without pipeline task aliases
# (works even when translation/text2text tasks are not registered).
mt_model_name = "Helsinki-NLP/opus-mt-en-es"
mt_tokenizer = AutoTokenizer.from_pretrained(mt_model_name)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(mt_model_name)

mt_inputs = mt_tokenizer(reviews, return_tensors="pt", padding=True, truncation=True)
with torch.no_grad():
    mt_output_ids = mt_model.generate(**mt_inputs, max_length=28)
translation = mt_tokenizer.batch_decode(mt_output_ids, skip_special_tokens=True)

first_translation = translation[0]
print(first_translation)

translated_review = first_translation

with open("reference_translations.txt", "r") as file:
    lines = file.readlines()
    references = [line.strip() for line in lines]

# Loading the BLEU metric

bleu = evaluate.load("bleu")
results = bleu.compute(predictions=[translated_review], references=[references])

bleu_score = results
print(f"BLEU Score: {bleu_score}")

model_ckpt = "deepset/minilm-uncased-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForQuestionAnswering.from_pretrained(model_ckpt)

# Preprocessing the input

context = reviews[1]
question = "What did he like about the brand?"

# rename `input` to `inputs` to avoid shadowing builtin
inputs = tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

start_logits = outputs.start_logits
end_logits = outputs.end_logits

# Post-processing the result
# assume batch size 1: argmax over sequence dimension and convert to int
start_idx = torch.argmax(start_logits, dim=-1).item()
end_idx = torch.argmax(end_logits, dim=-1).item()

answer_tokens = inputs["input_ids"][0][start_idx: end_idx + 1]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

print(f"Question: {question}")
print(f"Answer: {answer}")

# Summarizing the last review without pipeline task aliases
last_review = reviews[-1]

sum_model_name = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(sum_model_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_model_name)

sum_inputs = sum_tokenizer(last_review, return_tensors="pt", truncation=True, max_length=1024)
with torch.no_grad():
    sum_output_ids = sum_model.generate(
        **sum_inputs,
        max_length=80,
        min_length=30,
        num_beams=4,
        early_stopping=True,
    )

summarized_text = sum_tokenizer.decode(sum_output_ids[0], skip_special_tokens=True)
print(f"Summary: {summarized_text}")

# Analyzing for the potential biases

toxicity_metric = evaluate.load("toxicity")

toxicity_results = toxicity_metric.compute(predictions=[summarized_text])
# handle scalar or list-return formats
toxicity_values = toxicity_results.get('toxicity', toxicity_results)
if isinstance(toxicity_values, list):
    max_toxicity = max(toxicity_values)
else:
    max_toxicity = toxicity_values
print(f"Maximum Toxicity: {max_toxicity}")

regard_metric = evaluate.load("regard")

regard_result = regard_metric.compute(data=[summarized_text])
print(f"Regard Results: {regard_result}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2395.36it/s]


0.8 0.8571428571428571


Loading weights: 100%|██████████| 258/258 [00:00<00:00, 14611.54it/s]


Estoy muy satisfecho con mi 2014 Nissan NV SL. Uso esta furgoneta para mis entregas de negocios y uso personal. Camping
BLEU Score: {'bleu': 0.5741865870062566, 'precisions': [0.8695652173913043, 0.6818181818181818, 0.5238095238095238, 0.35], 'brevity_penalty': 1.0, 'length_ratio': 1.0952380952380953, 'translation_length': 23, 'reference_length': 21}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1010.50it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/minilm-uncased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Summary: The Nissan Rogue provides the desired SUV experience without burdening me with an exorbitant payment. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. The engine delivers strong performance, and the ride is really smooth.


Using default facebook/roberta-hate-speech-dynabench-r4-target checkpoint
c:\Users\Victus\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Victus\.cache\huggingface\hub\models--facebook--roberta-hate-speech-dynabench-r4-target. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.war

Maximum Toxicity: 0.00013919762568548322


c:\Users\Victus\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Victus\.cache\huggingface\hub\models--sasha--regardv3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 50217.72it/s]

Regard Results: {'regard': [[{'label': 'positive', 'score': 0.8488722443580627}, {'label': 'neutral', 'score': 0.06961768865585327}, {'label': 'other', 'score': 0.06699454039335251}, {'label': 'negative', 'score': 0.014515518210828304}]]}
